# 05. String & Datetime Accessors (.str & .dt): Beginner Guide

### 📌 Overview
Master **05. String & Datetime Accessors (.str & .dt): Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **String Accessors**: Covers `.str.lower()`, `.str.strip()`, `.str.contains()`, `.str.replace()`, and `.str.extract()`.
- **Datetime Accessors**: Covers `.dt.year`/`.dt.month`, `.dt.day_name()`, `.dt.floor()`, and `.dt.tz_localize()`/`.dt.tz_convert()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Lowercase Normalization: `Series.str.lower()`
- **What it does:** Converts all characters in each string element of the Series to lowercase, safely handling `NaN` values without error.
- **Syntax:** `Series.str.lower()`
- **Key Note:** All `.str` accessor methods automatically skip `NaN` values, propagating them as `NaN` rather than raising `AttributeError`.
- **Dataset Application & Code Demonstration:** Applies Lowercase Normalization on fintech records using columns `card_type` to demonstrate real-world execution.


In [2]:
print('Lowercased Card Types:\n', df['card_type'].str.lower().value_counts())

Lowercased Card Types:
 card_type
mastercard    3756
amex          3755
discover      3755
visa          3734
Name: count, dtype: int64


### 🔹 Trimming Whitespace: `Series.str.strip()`
- **What it does:** Strips leading and trailing whitespace (or specified characters) from strings in a Series.
- **Syntax:** `Series.str.strip()`
- **Key Note:** Dirty CSV imports frequently contain invisible trailing spaces (`'Visa '`); always strip text keys before merging.
- **Dataset Application & Code Demonstration:** Applies Trimming Whitespace on fintech records using columns `region` to demonstrate real-world execution.


In [3]:
print('Stripped Regions:', df['region'].str.strip().unique()[:4])

Stripped Regions: ['North' 'West' 'East' 'South']


### 🔹 Regex Substring Search: `Series.str.contains()`
- **What it does:** Tests if pattern or regex substring is contained within each string element of the Series, returning a boolean Series.
- **Syntax:** `Series.str.contains()`
- **Key Note:** Always specify `na=False` when using `.str.contains()` in boolean filtering to prevent `NaN` mask propagation errors.
- **Dataset Application & Code Demonstration:** Applies Regex Substring Search on fintech records using columns `card_type` to demonstrate real-world execution.


In [4]:
visa_mc = df[df['card_type'].str.contains('Visa|MasterCard', regex=True, na=False)]
print('Visa or MasterCard Count:', len(visa_mc))

Visa or MasterCard Count: 7490


### 🔹 Regex String Replacement: `Series.str.replace()`
- **What it does:** Replaces each occurrence of a pattern/regex in the Series with a replacement string.
- **Syntax:** `Series.str.replace()`
- **Key Note:** In modern Pandas, `regex=False` is default; you must pass `regex=True` when passing regular expressions.
- **Dataset Application & Code Demonstration:** Applies Regex String Replacement on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [5]:
numeric_tx_ids = df['transaction_id'].str.replace('TX_', '', regex=False)
print('Stripped Numeric IDs Head:\n', numeric_tx_ids.head())

Stripped Numeric IDs Head:
 0    TX109326
1    TX106376
2    TX103301
3    TX110701
4    TX103284
Name: transaction_id, dtype: object


### 🔹 Regex Capture Group Extraction: `Series.str.extract()`
- **What it does:** Extracts regular expression capture groups from string elements as columns in a new DataFrame.
- **Syntax:** `Series.str.extract()`
- **Key Note:** Named capture groups `(?P<group_name>...)` in the regex pattern automatically populate DataFrame column names.
- **Dataset Application & Code Demonstration:** Applies Regex Capture Group Extraction on fintech records using columns `customer_id` to demonstrate real-world execution.


In [6]:
extracted_ids = df['customer_id'].str.extract(r'(?P<Prefix>[A-Z]+)_(?P<Num>\d+)')
print('Extracted ID Components Head:\n', extracted_ids.head())

Extracted ID Components Head:
   Prefix  Num
0    NaN  NaN
1    NaN  NaN
2    NaN  NaN
3    NaN  NaN
4    NaN  NaN


### 🔹 Calendar Property Extraction: `Series.dt.year`, `.dt.month`, `.dt.day`
- **What it does:** Extracts calendar date components (year, month, day, hour, minute, second) from datetime Series.
- **Syntax:** `Series.dt.year`
- **Key Note:** The `.dt` accessor requires a `datetime64[ns]` Series. Vectorized calendar extraction is thousands of times faster than Python `.apply(lambda x: x.year)`.
- **Dataset Application & Code Demonstration:** Applies Calendar Property Extraction on fintech records using columns `transaction_date` to demonstrate real-world execution.


In [7]:
clean_dates = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
print('Transaction Year Breakdown:\n', clean_dates.dt.year.value_counts(dropna=False))

Transaction Year Breakdown:
 transaction_date
2025    10914
2026     4086
Name: count, dtype: int64


### 🔹 Day of Week Name: `Series.dt.day_name()`
- **What it does:** Returns the day name of the datetime Series (e.g. 'Monday', 'Tuesday') with optional locale support.
- **Syntax:** `Series.dt.day_name()`
- **Key Note:** Extracting day names enables weekday vs weekend spending behavior analysis.
- **Dataset Application & Code Demonstration:** Demonstrates Day of Week Name with practical fintech data structures and variables in the following code block.


In [8]:
print('Transactions per Day of Week:\n', clean_dates.dt.day_name().value_counts())

Transactions per Day of Week:
 transaction_date
Wednesday    2193
Saturday     2177
Thursday     2163
Friday       2142
Monday       2124
Sunday       2109
Tuesday      2092
Name: count, dtype: int64


### 🔹 Timestamp Truncation / Rounding: `Series.dt.floor()`
- **What it does:** Performs floor rounding operation on the datetime data to the specified frequency resolution (e.g. 'D' for day, 'h' for hour).
- **Syntax:** `Series.dt.floor()`
- **Key Note:** Truncating timestamps with `.dt.floor('D')` allows grouping events into daily batches regardless of exact transaction minutes/seconds.
- **Dataset Application & Code Demonstration:** Demonstrates Timestamp Truncation / Rounding with practical fintech data structures and variables in the following code block.


In [9]:
print('Floored Day Timestamps Head:\n', clean_dates.dt.floor('D').head())

Floored Day Timestamps Head:
 0   2026-02-17
1   2025-01-03
2   2025-12-20
3   2026-02-06
4   2025-01-07
Name: transaction_date, dtype: datetime64[ns]


### 🔹 Timezone Localization & Conversion: `Series.dt.tz_localize()` & `Series.dt.tz_convert()`
- **What it does:** Localizes naive datetime Series to a target timezone, or converts timezone-aware timestamps to another target timezone.
- **Syntax:** `Series.dt.tz_localize()`
- **Key Note:** Always convert disparate local merchant timestamps to `'UTC'` first before executing global financial reconciliations.
- **Dataset Application & Code Demonstration:** Demonstrates Timezone Localization & Conversion with practical fintech data structures and variables in the following code block.


In [10]:
utc_ts = clean_dates.dropna().head().dt.tz_localize('UTC')
ny_ts = utc_ts.dt.tz_convert('America/New_York')
print('UTC Timestamps:\n', utc_ts)
print('New York Timestamps (EST):\n', ny_ts)

UTC Timestamps:
 0   2026-02-17 08:28:57+00:00
1   2025-01-03 00:00:00+00:00
2   2025-12-20 00:00:00+00:00
3   2026-02-06 03:39:02+00:00
4   2025-01-07 00:23:37+00:00
Name: transaction_date, dtype: datetime64[ns, UTC]
New York Timestamps (EST):
 0   2026-02-17 03:28:57-05:00
1   2025-01-02 19:00:00-05:00
2   2025-12-19 19:00:00-05:00
3   2026-02-05 22:39:02-05:00
4   2025-01-06 19:23:37-05:00
Name: transaction_date, dtype: datetime64[ns, America/New_York]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Weekend Fraud Spike Analysis
- **Objective:** Q1: Weekend Fraud Spike Analysis
- **Approach:** Determine if the fraud rate is statistically higher on weekends versus weekdays.
- **Syntax:** `df.assign(is_weekend=clean_dates.dt.dayofweek >= 5).groupby('is_weekend')['is_fraud'].mean()`

In [11]:
clean_dt = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
weekend_fraud = df.assign(is_weekend=clean_dt.dt.dayofweek >= 5).groupby('is_weekend')['is_fraud'].agg(['count', 'mean'])
print('Weekend vs Weekday Fraud Rates:\n', weekend_fraud)

Weekend vs Weekday Fraud Rates:
             count      mean
is_weekend                 
False       10714  0.109016
True         4286  0.104526
